# Magnetic-one


Summary of the workflow diagram of **Magnetic-One Multi-Agent System** presented in its article:

### 1. Outer Loop – Initial Setup and Planning

   - **Task Trigger**: The process begins when a prompt or task is initiated. This prompts the **Orchestrator**, the main coordinating agent, to set up a **Task Ledger**.
   - **Task Ledger Creation**: The Task Ledger acts as a short-term memory for the task, recording key details that will guide the workflow.
   - **Fact and Guess Collection**: The Orchestrator gathers and pre-populates the Task Ledger with known facts, information to look up, derivable data, and educated guesses to frame initial responses.
   - **Plan Formation**: Using the Task Ledger and the capabilities of the available agents, the Orchestrator creates a **step-by-step plan**. This plan provides hints for task execution, guiding each agent in its respective role.
   - **Inner Loop Initiation**: After establishing the plan, the Orchestrator starts the **Inner Loop**.

### 2. Inner Loop – Iterative Task Execution and Monitoring

   During each iteration of the inner loop, the Orchestrator:
   
   - **Evaluates Task Completion**: Checks if the task has been fully satisfied.
   - **Checks for Loops or Stalls**: Monitors for repeated steps or a lack of forward progress.
   - **Makes Adjustments**: If progress is slow or looping is detected, the **Counter** (a stalling indicator) is incremented.
   - **Agent Selection and Instructions**: As long as progress continues or the Counter is within threshold limits, the Orchestrator selects the next agent to act and gives it specific instructions for the task at hand.
   - **Reflection and Self-Refinement**: If the Counter exceeds the threshold, the Orchestrator pauses the inner loop, revisits its previous steps, updates the Task Ledger, and revises the plan. This self-reflection allows the system to adapt and correct any identified issues before resuming the inner loop.

   The inner loop continues until the Orchestrator determines the task is complete or hits a predefined stopping criterion (e.g., maximum attempts or time limits).

### 3. Agents – Specialized Task Execution

   - **WebSurfer**: Manages interactions with a web browser, handling tasks like navigating websites, clicking, typing, or summarizing webpage content.
   - **FileSurfer**: Focuses on navigating and reading files, such as PDFs and images, to retrieve necessary data.
   - **Coder**: Develops, analyzes, or debugs code as needed for the task.
   - **ComputerTerminal**: Executes code or installs libraries to support programming tasks, interacting with a console environment.

   Each agent specializes in particular actions, and the Orchestrator directs agents as needed to accomplish the overall task.

### 4. Termination and Final Reporting

   - Once the task is complete or the termination conditions are met, both the outer and inner loops end.
   - **Final Review and Report**: The Orchestrator reviews all progress records and the Task Ledger to produce either a final solution or its best educated guess if uncertainties remain.

In summary, the workflow allows the Orchestrator to iteratively guide agents, adapt to challenges, and refine its approach in real time. This multi-layered control ensures robust task execution even in complex, dynamic environments.

## Workflow Diagram

The following diagram illustrates the workflow of the MagneticOne multi-agent system, breaking down the **Outer Loop**, **Inner Loop**, and **Agent** components. This visualization provides a structural overview of task progression, agent roles, and decision points within the system.


![MagneticOne Workflow Diagram](diagram_representation_workflow.svg)


### Mermaid Chart Structure

You can view the diagram directly and copy it to edit it using this [Mermaid Chart link](https://www.mermaidchart.com/app/projects/e0960e96-f27b-4905-b83b-a4741a09c128/diagrams/bcc71cc4-c0fe-4267-a797-99ec0491a8fb/version/v0.1/edit).

In addition, here's the structure from Mermaid Chart: 

```mermaid
flowchart TD
    subgraph OuterLoop ["Outer Loop"]
        Start["Initial Task Trigger"]
        Start -->|Creates| TaskLedger["Task Ledger"]
        TaskLedger -->|Pre-populates with| Facts["Facts, Lookups, Guesses"]
        Facts -->|Uses roles to form| Plan["Step-by-Step Plan"]
        Plan -->|Starts| InnerLoop
    end

    subgraph InnerLoop ["Inner Loop"]
        InnerLoop -->|Evaluate| CheckTask{"Is Task Complete?"}
        CheckTask -->|Yes| Terminate[Terminate]
        CheckTask -->|No| ProgressCheck{"Stalling or Looping?"}
        ProgressCheck -->|No| NextAgent["Choose Next Agent"]
        NextAgent --> InnerLoop
        ProgressCheck -->|Yes| Counter["Increment Counter"]
        Counter -->|Exceeds Threshold| Reflect["Reflection & Self-Refinement"]
        Reflect --> UpdateLedger["Update Task Ledger and Plan"]
        UpdateLedger --> InnerLoop
    end

    subgraph Agents ["Agents in Magnetic-One System"]
        Orchestrator -- Directs --> WebSurfer
        Orchestrator -- Directs --> FileSurfer
        Orchestrator -- Directs --> Coder
        Orchestrator -- Directs --> ComputerTerminal
    end

    Terminate --> Review["Review & Final Report"]
    OuterLoop --> Agents


## Code adaptation

This notebook demonstrates the **Magnetic-One** workflow implemented on **LangGraph 1.x**. Version `0.1.0` migrates the original `0.0.1` draft to the current LangGraph APIs and restructures the code into an installable package under `src/magnetic_one_langgraph/`, with a unit/integration test suite under `tests/`.

**What changed since `0.0.1`** (also addressing [issue #1](https://github.com/EmmanuelRTM/magnetic-one-langgraph/issues/1), `InvalidUpdateError: Must write to at least one of []`):

- The graph state is now a `TypedDict` (`TaskState`) with **reducers** (`operator.add` for `messages`, a dict-merge for `task_ledger`), instead of a plain Python class. LangGraph requires a proper state schema so each key becomes a channel; the old class produced a schema with no writable channels, which is exactly what raised the `InvalidUpdateError`.
- Nodes return **partial state updates** instead of mutating and returning the whole state object.
- The Orchestrator routes dynamically with `langgraph.types.Command`, implementing the real inner loop from the paper: completion evaluation, stall detection with a counter, reflection/re-planning with a bounded replan budget, and a final report (or best educated guess) at termination.
- Workers (`WebSurfer`, `FileSurfer`, `Coder`, `ComputerTerminal`) always hand control back to the Orchestrator rather than being wired in a fixed linear pipeline.
- No API keys are required to run this scaffold — the agents are deterministic placeholders. **Agents may need tool calling** for real work; see [Hierarchical Agent Teams](https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/multi_agent/hierarchical_agent_teams.ipynb) and replace the agent bodies in `src/magnetic_one_langgraph/agents.py` while keeping the same partial-update contract.


### 1. Installation

Install the package (editable) together with its test dependencies. Run this notebook from the repository root.

In [ ]:
%%capture --no-stderr
%pip install -e ".[test]"

### 2. Optional: API keys

The scaffold runs fully offline. Keys are only needed once you wire real LLM-backed agents or tools (e.g. `pip install -e ".[llm]"` for `langchain` + `langchain-openai`).

In [ ]:
import getpass
import os

def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Please provide your {var}")

# Uncomment when plugging in LLM-backed agents or tools:
# _set_if_undefined("OPENAI_API_KEY")
# _set_if_undefined("TAVILY_API_KEY")

### 3. Build the workflow graph

The compiled graph mirrors the diagram above: the Orchestrator sits in the middle of the inner loop and every worker reports back to it.

In [ ]:
from magnetic_one_langgraph import build_graph

graph = build_graph()
print(graph.get_graph().draw_mermaid())

### 4. Run the task system

`run_task_system` seeds the initial state, streams orchestration messages as they happen, and returns the final state.

In [ ]:
from magnetic_one_langgraph import run_task_system

final_state = run_task_system("Analyze new market trends and compile a report.")

### 5. Inspect the final state

The Task Ledger collects everything the agents produced; `final_report` summarizes it.

In [ ]:
final_state["task_ledger"]

### 6. Running the tests

The behaviors above are covered by unit and integration tests (state reducers, orchestrator loop logic including stall/reflection paths, each agent, and the end-to-end graph run — including a regression test for issue #1):

```bash
pytest -v
```
